<a href="https://colab.research.google.com/github/halimAhtasham/Algorithm_Exercise/blob/main/config.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 01. CONFIGURATION & REPRODUCIBILITY
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

# -----------------------------
# Random seed
# -----------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Make experiments as reproducible as possible
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -----------------------------
# Device
# -----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cpu
Device: cpu


In [2]:
# ============================================================
# 02. REQUIRED LIBRARIES
# ============================================================

!pip -q install shap scikit-learn scipy pandas numpy matplotlib seaborn

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
# ============================================================
# 03. LOAD DATASET
# ============================================================
DATA_PATH = "/content/drive/MyDrive/Datasets/ROSIDS23.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (136681, 84)

Columns:
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'B

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.3.4-192.168.3.6-11311-60792-6,192.168.3.6,60792,192.168.3.4,11311,6,07/07/2023 02:10:23 PM,6260,5,5,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,192.168.3.4-192.168.3.6-11311-60794-6,192.168.3.6,60794,192.168.3.4,11311,6,07/07/2023 02:10:23 PM,5903,5,5,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,192.168.3.4-192.168.3.6-11311-39922-6,192.168.3.6,39922,192.168.3.4,11311,6,07/07/2023 02:10:32 PM,4523,5,5,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,192.168.3.4-192.168.3.6-11311-55266-6,192.168.3.6,55266,192.168.3.4,11311,6,07/07/2023 02:11:11 PM,5191,5,5,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,192.168.3.6-192.168.3.7-43770-11111-6,192.168.3.7,11111,192.168.3.6,43770,6,07/07/2023 02:10:03 PM,72625778,2200,2212,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


In [12]:
# ============================================
# STEP 4.1 — Dataset Inspection
# ============================================

data = df.copy()

print("Original shape:", data.shape)

print("\nLabel distribution:")
print(data["Label"].value_counts())

print("\nMissing values:")
print(data.isnull().sum().sum())

print("\nInfinite values:")
print(np.isinf(data.select_dtypes(include=np.number)).sum().sum())

Original shape: (136681, 84)

Label distribution:
Label
Benign       62511
DoS          31000
Subflood     30064
UnauthPub     7817
UnauthSub     5289
Name: count, dtype: int64

Missing values:
272

Infinite values:
278


In [13]:
# ============================================
# STEP 4.2 — Remove Non-Predictive Identifiers
# ============================================

# Columns that should NOT be used as ML features
identifier_columns = [
    "Flow ID",
    "Src IP",
    "Dst IP",
    "Timestamp"
]

data = data.drop(columns=identifier_columns)

print("Shape after removing identifiers:", data.shape)

print("\nRemaining columns:")
print(data.columns.tolist())

Shape after removing identifiers: (136681, 80)

Remaining columns:
['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg

In [14]:
# ============================================
# STEP 4.3 — Convert to Binary Classification
# ============================================

# Benign = 0
# Any other class = Attack = 1

data["Binary_Label"] = (data["Label"] != "Benign").astype(int)

# Remove original multiclass label
data = data.drop(columns=["Label"])

print("Binary label distribution:")
print(data["Binary_Label"].value_counts())

print("\nFinal dataset shape:", data.shape)

Binary label distribution:
Binary_Label
1    74170
0    62511
Name: count, dtype: int64

Final dataset shape: (136681, 80)


In [15]:
# ============================================
# STEP 4.4 — Handle Infinite Values
# ============================================

# Convert +inf and -inf into NaN
data = data.replace([np.inf, -np.inf], np.nan)

print("Total missing values after replacing infinity:")
print(data.isnull().sum().sum())

Total missing values after replacing infinity:
550


In [16]:
missing_columns = data.isnull().sum()

missing_columns = missing_columns[missing_columns > 0]

print("Columns containing missing values:")
print(missing_columns)

Columns containing missing values:
Flow Byts/s    275
Flow Pkts/s    275
dtype: int64


In [17]:
# ============================================
# STEP 5.1 — Separate Features and Target
# ============================================

X = data.drop(columns=["Binary_Label"])
y = data["Binary_Label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (136681, 79)
y shape: (136681,)


In [18]:
# ============================================
# STEP 5.2 — First Train/Test Split
# ============================================

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=SEED
)

print("Training set:", X_train.shape)
print("Temporary set:", X_temp.shape)

Training set: (95676, 79)
Temporary set: (41005, 79)


In [19]:
# ============================================
# STEP 5.3 — Validation/Test Split
# ============================================

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=SEED
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (95676, 79)
Validation: (20502, 79)
Test: (20503, 79)


In [20]:
# ============================================
# STEP 5.4 — Verify Stratification
# ============================================

print("\nTrain attack ratio:", y_train.mean())
print("Validation attack ratio:", y_val.mean())
print("Test attack ratio:", y_test.mean())

print("\nClass counts:")
print("Train:")
print(y_train.value_counts())

print("\nValidation:")
print(y_val.value_counts())

print("\nTest:")
print(y_test.value_counts())


Train attack ratio: 0.5426543751829089
Validation attack ratio: 0.5426299873183104
Test attack ratio: 0.5426522947861289

Class counts:
Train:
Binary_Label
1    51919
0    43757
Name: count, dtype: int64

Validation:
Binary_Label
1    11125
0     9377
Name: count, dtype: int64

Test:
Binary_Label
1    11126
0     9377
Name: count, dtype: int64


In [21]:
# ============================================
# STEP 6.1 — Train-Only Missing Value Imputation
# ============================================

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

# FIT ONLY on training data
X_train_imputed = imputer.fit_transform(X_train)

# Transform validation and test using training statistics
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

print("Missing values after imputation:")
print("Train:", np.isnan(X_train_imputed).sum())
print("Validation:", np.isnan(X_val_imputed).sum())
print("Test:", np.isnan(X_test_imputed).sum())

Missing values after imputation:
Train: 0
Validation: 0
Test: 0


In [22]:
# ============================================
# STEP 6.2 — Train-Only Feature Scaling
# ============================================

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# FIT ONLY on training data
X_train_scaled = scaler.fit_transform(X_train_imputed)

# Transform validation and test
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Scaled shapes:")
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

Scaled shapes:
Train: (95676, 79)
Validation: (20502, 79)
Test: (20503, 79)


In [23]:
# ============================================
# STEP 6.3 — Final Preprocessing Check
# ============================================

print("Train min:", X_train_scaled.min())
print("Train max:", X_train_scaled.max())

print("\nValidation min:", X_val_scaled.min())
print("Validation max:", X_val_scaled.max())

print("\nTest min:", X_test_scaled.min())
print("Test max:", X_test_scaled.max())

print("\nAny NaN?")
print("Train:", np.isnan(X_train_scaled).any())
print("Validation:", np.isnan(X_val_scaled).any())
print("Test:", np.isnan(X_test_scaled).any())

Train min: 0.0
Train max: 1.0

Validation min: 0.0
Validation max: 2.0

Test min: 0.0
Test max: 1.1697259168960004

Any NaN?
Train: False
Validation: False
Test: False


In [24]:
# ============================================
# STEP 7.1 — Convert Data to PyTorch
# ============================================

import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Validation:", X_val_tensor.shape, y_val_tensor.shape)
print("Test:", X_test_tensor.shape, y_test_tensor.shape)

Train: torch.Size([95676, 79]) torch.Size([95676, 1])
Validation: torch.Size([20502, 79]) torch.Size([20502, 1])
Test: torch.Size([20503, 79]) torch.Size([20503, 1])


In [25]:
# ============================================
# STEP 7.2 — Create DataLoaders
# ============================================

BATCH_SIZE = 256

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of test batches:", len(test_loader))

Number of training batches: 374
Number of validation batches: 81
Number of test batches: 81


In [26]:
# ============================================
# STEP 7.3 — Neural IDS Model
# ============================================

import torch.nn as nn

class IDSModel(nn.Module):

    def __init__(self, input_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


input_features = X_train_scaled.shape[1]

ids_model = IDSModel(input_features).to(DEVICE)

print(ids_model)

IDSModel(
  (network): Sequential(
    (0): Linear(in_features=79, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [27]:
# ============================================
# STEP 7.4 — Loss and Optimizer
# ============================================

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    ids_model.parameters(),
    lr=0.001
)

print("Loss:", criterion)
print("Optimizer:", optimizer)

Loss: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [28]:
# ============================================
# STEP 7.5 — Training Function
# ============================================

def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0.0

    for features, labels in loader:

        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(features)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * features.size(0)

    average_loss = total_loss / len(loader.dataset)

    return average_loss

In [29]:
# ============================================
# STEP 7.6 — Validation Function
# ============================================

def evaluate_loss(model, loader, criterion, device):

    model.eval()

    total_loss = 0.0

    with torch.no_grad():

        for features, labels in loader:

            features = features.to(device)
            labels = labels.to(device)

            logits = model(features)

            loss = criterion(logits, labels)

            total_loss += loss.item() * features.size(0)

    average_loss = total_loss / len(loader.dataset)

    return average_loss

In [30]:
# ============================================
# STEP 7.7 — Train IDS
# ============================================

EPOCHS = 20

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        ids_model,
        train_loader,
        criterion,
        optimizer,
        DEVICE
    )

    val_loss = evaluate_loss(
        ids_model,
        val_loader,
        criterion,
        DEVICE
    )

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch 01/20 | Train Loss: 0.2956 | Val Loss: 0.2253
Epoch 02/20 | Train Loss: 0.2235 | Val Loss: 0.2190
Epoch 03/20 | Train Loss: 0.2169 | Val Loss: 0.2141
Epoch 04/20 | Train Loss: 0.2133 | Val Loss: 0.2112
Epoch 05/20 | Train Loss: 0.2114 | Val Loss: 0.2106
Epoch 06/20 | Train Loss: 0.2081 | Val Loss: 0.2077
Epoch 07/20 | Train Loss: 0.2057 | Val Loss: 0.2055
Epoch 08/20 | Train Loss: 0.2024 | Val Loss: 0.1949
Epoch 09/20 | Train Loss: 0.1965 | Val Loss: 0.1842
Epoch 10/20 | Train Loss: 0.1726 | Val Loss: 0.1434
Epoch 11/20 | Train Loss: 0.1557 | Val Loss: 0.1289
Epoch 12/20 | Train Loss: 0.1436 | Val Loss: 0.1181
Epoch 13/20 | Train Loss: 0.1327 | Val Loss: 0.1182
Epoch 14/20 | Train Loss: 0.1286 | Val Loss: 0.1142
Epoch 15/20 | Train Loss: 0.1227 | Val Loss: 0.1159
Epoch 16/20 | Train Loss: 0.1206 | Val Loss: 0.1125
Epoch 17/20 | Train Loss: 0.1220 | Val Loss: 0.1126
Epoch 18/20 | Train Loss: 0.1242 | Val Loss: 0.1150
Epoch 19/20 | Train Loss: 0.1230 | Val Loss: 0.1094
Epoch 20/20 

In [31]:
# ============================================
# STEP 8.1 — Generate Test Predictions
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

def get_predictions(model, loader, device):

    model.eval()

    all_probabilities = []
    all_labels = []

    with torch.no_grad():

        for features, labels in loader:

            features = features.to(device)

            logits = model(features)

            probabilities = torch.sigmoid(logits)

            all_probabilities.extend(
                probabilities.cpu().numpy().flatten()
            )

            all_labels.extend(
                labels.numpy().flatten()
            )

    return np.array(all_labels), np.array(all_probabilities)


y_test_true, y_test_probability = get_predictions(
    ids_model,
    test_loader,
    DEVICE
)

y_test_pred = (y_test_probability >= 0.5).astype(int)

In [32]:
# ============================================
# STEP 8.2 — Clean IDS Baseline Metrics
# ============================================

accuracy = accuracy_score(y_test_true, y_test_pred)

precision = precision_score(y_test_true, y_test_pred)

recall = recall_score(y_test_true, y_test_pred)

f1 = f1_score(y_test_true, y_test_pred)

roc_auc = roc_auc_score(
    y_test_true,
    y_test_probability
)

pr_auc = average_precision_score(
    y_test_true,
    y_test_probability
)

print("========== CLEAN IDS BASELINE ==========")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

========== CLEAN IDS BASELINE ==========
Accuracy : 0.9685
Precision: 0.9761
Recall   : 0.9656
F1 Score : 0.9708
ROC-AUC  : 0.9885
PR-AUC   : 0.9925


In [33]:
# ============================================
# STEP 8.3 — Confusion Matrix
# ============================================

cm = confusion_matrix(
    y_test_true,
    y_test_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[ 9114   263]
 [  383 10743]]


In [34]:
# ============================================
# STEP 9.1 — Extract Hidden Representation
# ============================================

class IDSFeatureExtractor(nn.Module):

    def __init__(self, trained_model):
        super().__init__()

        self.feature_network = nn.Sequential(
            *list(trained_model.network.children())[:-1]
        )

    def forward(self, x):
        return self.feature_network(x)


feature_extractor = IDSFeatureExtractor(ids_model).to(DEVICE)

feature_extractor.eval()

print(feature_extractor)

IDSFeatureExtractor(
  (feature_network): Sequential(
    (0): Linear(in_features=79, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
  )
)


In [35]:
# ============================================
# STEP 9.2 — Correctly Classified Benign Data
# ============================================

ids_model.eval()

with torch.no_grad():

    train_logits = ids_model(X_train_tensor.to(DEVICE))

    train_probabilities = torch.sigmoid(train_logits).cpu().numpy().flatten()

train_predictions = (train_probabilities >= 0.5).astype(int)

train_labels = y_train.values

correct_benign_mask = (
    (train_labels == 0) &
    (train_predictions == 0)
)

X_benign_correct = X_train_tensor[correct_benign_mask]

print("Total benign training samples:",
      (train_labels == 0).sum())

print("Correctly classified benign samples:",
      correct_benign_mask.sum())

print("Reconstruction input shape:",
      X_benign_correct.shape)

Total benign training samples: 43757
Correctly classified benign samples: 42682
Reconstruction input shape: torch.Size([42682, 79])


In [36]:
# ============================================
# STEP 9.3 — Extract Benign Representations
# ============================================

with torch.no_grad():

    benign_representations = feature_extractor(
        X_benign_correct.to(DEVICE)
    ).cpu().numpy()

print("Benign representation shape:",
      benign_representations.shape)

Benign representation shape: (42682, 32)


In [55]:
# ============================================
# STEP 10 — READ Reconstruction Model
# ============================================

from sklearn.linear_model import LinearRegression

# Get correctly classified benign TRAINING samples
correct_benign_mask = (
    (y_train.values == 0) &
    (train_predictions == 0)
)

X_benign_correct = X_train_scaled[correct_benign_mask]

print("Correctly classified benign samples:", X_benign_correct.shape)

# Extract 32-D hidden representations
ids_model.eval()

with torch.no_grad():
    benign_representation_tensor = feature_extractor(
        torch.tensor(X_benign_correct, dtype=torch.float32).to(DEVICE)
    )

benign_representations = benign_representation_tensor.cpu().numpy()

print("Hidden representation shape:", benign_representations.shape)
print("Original feature shape:", X_benign_correct.shape)

Correctly classified benign samples: (42682, 79)
Hidden representation shape: (42682, 32)
Original feature shape: (42682, 79)


In [56]:
# ============================================
# STEP 10.1 — Reconstruction Train/Validation
# ============================================

from sklearn.model_selection import train_test_split

benign_rep_train, benign_rep_val, \
X_original_train, X_original_val = train_test_split(
    benign_representations,
    X_benign_correct,
    test_size=0.20,
    random_state=SEED
)

print("Reconstruction training:")
print("Hidden:", benign_rep_train.shape)
print("Original:", X_original_train.shape)

print("\nReconstruction validation:")
print("Hidden:", benign_rep_val.shape)
print("Original:", X_original_val.shape)

Reconstruction training:
Hidden: (34145, 32)
Original: (34145, 79)

Reconstruction validation:
Hidden: (8537, 32)
Original: (8537, 79)


In [58]:
# ============================================
# STEP 10.2 — Differentiable READ Reconstructor
# ============================================

import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class ReconstructionModel(nn.Module):
    def __init__(self, hidden_dim=32, output_dim=79):
        super().__init__()

        self.reconstructor = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, hidden_representation):
        return self.reconstructor(hidden_representation)


reconstruction_model = ReconstructionModel(
    hidden_dim=32,
    output_dim=79
).to(DEVICE)

reconstruction_optimizer = torch.optim.Adam(
    reconstruction_model.parameters(),
    lr=0.001
)

reconstruction_criterion = nn.MSELoss()

RECONSTRUCTION_EPOCHS = 30
BATCH_SIZE = 256

In [59]:
# ============================================
# STEP 10.3 — Reconstruction DataLoaders
# ============================================

rep_train_tensor = torch.tensor(
    benign_rep_train,
    dtype=torch.float32
)

original_train_tensor = torch.tensor(
    X_original_train,
    dtype=torch.float32
)

rep_val_tensor = torch.tensor(
    benign_rep_val,
    dtype=torch.float32
)

original_val_tensor = torch.tensor(
    X_original_val,
    dtype=torch.float32
)

reconstruction_train_dataset = TensorDataset(
    rep_train_tensor,
    original_train_tensor
)

reconstruction_val_dataset = TensorDataset(
    rep_val_tensor,
    original_val_tensor
)

reconstruction_train_loader = DataLoader(
    reconstruction_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

reconstruction_val_loader = DataLoader(
    reconstruction_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [60]:
# ============================================
# STEP 10.4 — Train Reconstruction Model
# ============================================

best_val_loss = float("inf")
best_state = None

for epoch in range(RECONSTRUCTION_EPOCHS):

    # -------------------------
    # Training
    # -------------------------
    reconstruction_model.train()

    train_loss = 0.0

    for hidden_batch, original_batch in reconstruction_train_loader:

        hidden_batch = hidden_batch.to(DEVICE)
        original_batch = original_batch.to(DEVICE)

        reconstruction_optimizer.zero_grad()

        reconstructed = reconstruction_model(hidden_batch)

        loss = reconstruction_criterion(
            reconstructed,
            original_batch
        )

        loss.backward()
        reconstruction_optimizer.step()

        train_loss += loss.item() * hidden_batch.size(0)

    train_loss /= len(reconstruction_train_dataset)

    # -------------------------
    # Validation
    # -------------------------
    reconstruction_model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for hidden_batch, original_batch in reconstruction_val_loader:

            hidden_batch = hidden_batch.to(DEVICE)
            original_batch = original_batch.to(DEVICE)

            reconstructed = reconstruction_model(
                hidden_batch
            )

            loss = reconstruction_criterion(
                reconstructed,
                original_batch
            )

            val_loss += loss.item() * hidden_batch.size(0)

    val_loss /= len(reconstruction_val_dataset)

    # Save best model
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_state = {
            key: value.cpu().clone()
            for key, value
            in reconstruction_model.state_dict().items()
        }

    if (epoch + 1) % 5 == 0 or epoch == 0:

        print(
            f"Epoch {epoch+1:02d}/{RECONSTRUCTION_EPOCHS} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f}"
        )


# Load best model
reconstruction_model.load_state_dict(best_state)
reconstruction_model.to(DEVICE)

print("\nBest validation loss:", best_val_loss)

Epoch 01/30 | Train Loss: 1.018634 | Val Loss: 0.106269
Epoch 05/30 | Train Loss: 0.033599 | Val Loss: 0.033345
Epoch 10/30 | Train Loss: 0.033096 | Val Loss: 0.033023
Epoch 15/30 | Train Loss: 0.032576 | Val Loss: 0.032812
Epoch 20/30 | Train Loss: 0.032257 | Val Loss: 0.032123
Epoch 25/30 | Train Loss: 0.031924 | Val Loss: 0.031645
Epoch 30/30 | Train Loss: 0.032748 | Val Loss: 0.032346

Best validation loss: 0.031507398579276195


In [61]:
# ============================================
# STEP 11 — Validate READ Reconstruction
# ============================================

reconstruction_model.eval()

with torch.no_grad():

    reconstructed_val = reconstruction_model(
        rep_val_tensor.to(DEVICE)
    ).cpu().numpy()

original_val = original_val_tensor.numpy()

# Absolute error for every feature
absolute_error = np.abs(
    original_val - reconstructed_val
)

# MedAE for every sample
benign_medae = np.median(
    absolute_error,
    axis=1
)

print("READ Reconstruction Error — Clean Benign Validation")
print("=" * 55)

print(f"Number of samples: {len(benign_medae)}")

print(f"\nMean   : {benign_medae.mean():.6f}")
print(f"Median : {np.median(benign_medae):.6f}")
print(f"Std    : {benign_medae.std():.6f}")
print(f"Min    : {benign_medae.min():.6f}")
print(f"Max    : {benign_medae.max():.6f}")

print("\nPercentiles:")

for percentile in [90, 95, 99, 99.5, 99.9]:

    print(
        f"{percentile}% : "
        f"{np.percentile(benign_medae, percentile):.6f}"
    )

READ Reconstruction Error — Clean Benign Validation
Number of samples: 8537

Mean   : 0.027902
Median : 0.023153
Std    : 0.013479
Min    : 0.002987
Max    : 0.472209

Percentiles:
90% : 0.045483
95% : 0.056070
99% : 0.070346
99.5% : 0.088910
99.9% : 0.149313


In [62]:
# ============================================
# STEP 12A — READ Uncertainty Functions
# ============================================

import numpy as np
import torch
from scipy.stats import entropy as scipy_entropy


def calculate_medae(original_data, reconstructed_data):
    """
    Calculate Median Absolute Error for each sample.
    """
    absolute_error = np.abs(
        original_data - reconstructed_data
    )

    return np.median(
        absolute_error,
        axis=1
    )


def calculate_mc_dropout_uncertainty(
    model,
    X,
    n_passes=20
):
    """
    Calculate:
    - Aleatoric uncertainty
    - Epistemic uncertainty
    - Predictive entropy

    using MC Dropout.
    """

    # Enable dropout during inference
    model.train()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    ).to(DEVICE)

    probability_samples = []

    with torch.no_grad():

        for _ in range(n_passes):

            logits = model(X_tensor)

            probabilities = torch.sigmoid(
                logits
            ).squeeze(1)

            probability_samples.append(
                probabilities.cpu().numpy()
            )

    probability_samples = np.stack(
        probability_samples,
        axis=0
    )

    # Mean prediction
    mean_probability = (
        probability_samples.mean(axis=0)
    )

    # Epistemic uncertainty
    epistemic_uncertainty = (
        probability_samples.var(axis=0)
    )

    # Aleatoric uncertainty
    aleatoric_uncertainty = (
        mean_probability *
        (1.0 - mean_probability)
    )

    # Predictive entropy
    probability_matrix = np.column_stack([
        1.0 - mean_probability,
        mean_probability
    ])

    predictive_entropy = scipy_entropy(
        probability_matrix.T + 1e-12,
        axis=0
    )

    # Return model to evaluation mode
    model.eval()

    return (
        aleatoric_uncertainty,
        epistemic_uncertainty,
        predictive_entropy
    )


print("Step 12A completed.")

Step 12A completed.


In [63]:
# ============================================
# STEP 12B — READ Metrics for Clean Benign
# ============================================

# --------------------------------------------
# 1. Original benign validation samples
# --------------------------------------------

X_benign_read = X_original_val

print("Input shape:", X_benign_read.shape)


# --------------------------------------------
# 2. Calculate MedAE
# --------------------------------------------

# reconstructed_val was generated in Step 11

benign_medae = calculate_medae(
    X_original_val,
    reconstructed_val
)


# --------------------------------------------
# 3. Calculate MC Dropout uncertainties
# --------------------------------------------

(
    benign_aleatoric,
    benign_epistemic,
    benign_entropy
) = calculate_mc_dropout_uncertainty(
    ids_model,
    X_benign_read,
    n_passes=20
)


# --------------------------------------------
# 4. Check shapes
# --------------------------------------------

print("\nMetric shapes:")
print("MedAE:", benign_medae.shape)
print("Aleatoric:", benign_aleatoric.shape)
print("Epistemic:", benign_epistemic.shape)
print("Entropy:", benign_entropy.shape)


# --------------------------------------------
# 5. Print statistics
# --------------------------------------------

print("\nREAD metrics — Clean Benign Samples")
print("=" * 55)

print(f"Number of samples: {len(X_benign_read)}")


print("\nMedAE:")
print(f"  Mean   : {benign_medae.mean():.6f}")
print(f"  Median : {np.median(benign_medae):.6f}")
print(f"  Std    : {benign_medae.std():.6f}")
print(f"  Min    : {benign_medae.min():.6f}")
print(f"  Max    : {benign_medae.max():.6f}")


print("\nAleatoric Uncertainty:")
print(f"  Mean   : {benign_aleatoric.mean():.6f}")
print(f"  Median : {np.median(benign_aleatoric):.6f}")
print(f"  Std    : {benign_aleatoric.std():.6f}")
print(f"  Min    : {benign_aleatoric.min():.6f}")
print(f"  Max    : {benign_aleatoric.max():.6f}")


print("\nEpistemic Uncertainty:")
print(f"  Mean   : {benign_epistemic.mean():.6f}")
print(f"  Median : {np.median(benign_epistemic):.6f}")
print(f"  Std    : {benign_epistemic.std():.6f}")
print(f"  Min    : {benign_epistemic.min():.6f}")
print(f"  Max    : {benign_epistemic.max():.6f}")


print("\nPredictive Entropy:")
print(f"  Mean   : {benign_entropy.mean():.6f}")
print(f"  Median : {np.median(benign_entropy):.6f}")
print(f"  Std    : {benign_entropy.std():.6f}")
print(f"  Min    : {benign_entropy.min():.6f}")
print(f"  Max    : {benign_entropy.max():.6f}")


print("\nStep 12B completed.")

Input shape: (8537, 79)

Metric shapes:
MedAE: (8537,)
Aleatoric: (8537,)
Epistemic: (8537,)
Entropy: (8537,)

READ metrics — Clean Benign Samples
Number of samples: 8537

MedAE:
  Mean   : 0.027902
  Median : 0.023153
  Std    : 0.013479
  Min    : 0.002987
  Max    : 0.472209

Aleatoric Uncertainty:
  Mean   : 0.037626
  Median : 0.033772
  Std    : 0.029307
  Min    : 0.000000
  Max    : 0.249985

Epistemic Uncertainty:
  Mean   : 0.003977
  Median : 0.000144
  Std    : 0.009324
  Min    : 0.000000
  Max    : 0.116689

Predictive Entropy:
  Mean   : 0.156339
  Median : 0.151703
  Std    : 0.094577
  Min    : 0.000000
  Max    : 0.693117

Step 12B completed.


In [73]:
# ============================================
# STEP 13 — Correct Clean Attack Samples
# ============================================

# Use the PROCESSED + SCALED validation data
X_val_processed = X_val_scaled

y_val_array = (
    y_val.to_numpy()
    if hasattr(y_val, "to_numpy")
    else y_val
)

# Select attack samples
attack_mask_val = (y_val_array == 1)

X_attack_clean = X_val_processed[attack_mask_val]

print("Clean attack samples:", X_attack_clean.shape)

print(
    "NaN values:",
    np.isnan(X_attack_clean).sum()
)

print(
    "Infinite values:",
    np.isinf(X_attack_clean).sum()
)

Clean attack samples: (11125, 79)
NaN values: 0
Infinite values: 0


In [74]:
# ============================================
# STEP 13.1 — Extract Attack Representations
# ============================================

ids_model.eval()

X_attack_tensor = torch.tensor(
    X_attack_clean,
    dtype=torch.float32
).to(DEVICE)

with torch.no_grad():

    attack_representation_tensor = feature_extractor(
        X_attack_tensor
    )

attack_representations = (
    attack_representation_tensor
    .cpu()
    .numpy()
)

print("Attack input shape:", X_attack_clean.shape)
print(
    "Attack representation shape:",
    attack_representations.shape
)

print(
    "Representation NaN:",
    np.isnan(attack_representations).sum()
)

Attack input shape: (11125, 79)
Attack representation shape: (11125, 32)
Representation NaN: 0


In [72]:
# ============================================
# STEP 13 FIX — Check Validation Data
# ============================================

print("X_val NaN:", X_val.isna().sum().sum())
print("X_val Inf:", np.isinf(X_val.select_dtypes(include=np.number)).sum().sum())

print("\nX_val_imputed NaN:", np.isnan(X_val_imputed).sum())
print("X_val_imputed Inf:", np.isinf(X_val_imputed).sum())

print("\nX_val_scaled NaN:", np.isnan(X_val_scaled).sum())
print("X_val_scaled Inf:", np.isinf(X_val_scaled).sum())

X_val NaN: 62
X_val Inf: 0

X_val_imputed NaN: 0
X_val_imputed Inf: 0

X_val_scaled NaN: 0
X_val_scaled Inf: 0


In [76]:
# ============================================
# STEP 13.2 — READ Metrics for Clean Attacks
# ============================================

# --------------------------------------------
# 1. Reconstruct attack samples
# --------------------------------------------

reconstruction_model.eval()

attack_rep_tensor = torch.tensor(
    attack_representations,
    dtype=torch.float32
).to(DEVICE)

with torch.no_grad():

    attack_reconstructed = reconstruction_model(
        attack_rep_tensor
    ).cpu().numpy()


# --------------------------------------------
# 2. Calculate MedAE
# --------------------------------------------

attack_medae = calculate_medae(
    X_attack_clean,
    attack_reconstructed
)


# --------------------------------------------
# 3. Calculate uncertainty metrics
# --------------------------------------------

(
    attack_aleatoric,
    attack_epistemic,
    attack_entropy
) = calculate_mc_dropout_uncertainty(
    ids_model,
    X_attack_clean,
    n_passes=20
)


# --------------------------------------------
# 4. Check for invalid values
# --------------------------------------------

print("NaN check:")
print("MedAE:", np.isnan(attack_medae).sum())
print("Aleatoric:", np.isnan(attack_aleatoric).sum())
print("Epistemic:", np.isnan(attack_epistemic).sum())
print("Entropy:", np.isnan(attack_entropy).sum())


# --------------------------------------------
# 5. Print statistics
# --------------------------------------------

print("\nREAD metrics — Clean Attack Samples")
print("=" * 55)

print(f"Number of samples: {len(X_attack_clean)}")

print("\nMedAE:")
print(f"  Mean   : {attack_medae.mean():.6f}")
print(f"  Median : {np.median(attack_medae):.6f}")
print(f"  Std    : {attack_medae.std():.6f}")

print("\nAleatoric Uncertainty:")
print(f"  Mean   : {attack_aleatoric.mean():.6f}")
print(f"  Median : {np.median(attack_aleatoric):.6f}")
print(f"  Std    : {attack_aleatoric.std():.6f}")

print("\nEpistemic Uncertainty:")
print(f"  Mean   : {attack_epistemic.mean():.6f}")
print(f"  Median : {np.median(attack_epistemic):.6f}")
print(f"  Std    : {attack_epistemic.std():.6f}")

print("\nPredictive Entropy:")
print(f"  Mean   : {attack_entropy.mean():.6f}")
print(f"  Median : {np.median(attack_entropy):.6f}")
print(f"  Std    : {attack_entropy.std():.6f}")

NaN check:
MedAE: 0
Aleatoric: 0
Epistemic: 0
Entropy: 0

READ metrics — Clean Attack Samples
Number of samples: 11125

MedAE:
  Mean   : 0.180632
  Median : 0.141953
  Std    : 0.137671

Aleatoric Uncertainty:
  Mean   : 0.027463
  Median : 0.000002
  Std    : 0.057737

Epistemic Uncertainty:
  Mean   : 0.002621
  Median : 0.000000
  Std    : 0.009674

Predictive Entropy:
  Mean   : 0.090313
  Median : 0.000025
  Std    : 0.179349


In [77]:
# ============================================
# STEP 14 — Organize READ Metric Data
# ============================================

import numpy as np

# Clean benign metrics
benign_features = np.column_stack([
    benign_medae,
    benign_aleatoric,
    benign_epistemic,
    benign_entropy
])

# Clean attack metrics
attack_features = np.column_stack([
    attack_medae,
    attack_aleatoric,
    attack_epistemic,
    attack_entropy
])

print("Benign READ feature shape:", benign_features.shape)
print("Attack READ feature shape:", attack_features.shape)

print("\nFeature order:")
print([
    "MedAE",
    "Aleatoric",
    "Epistemic",
    "Entropy"
])

Benign READ feature shape: (8537, 4)
Attack READ feature shape: (11125, 4)

Feature order:
['MedAE', 'Aleatoric', 'Epistemic', 'Entropy']


In [84]:
# ============================================
# STEP 15 — Correct FGSM Oblivious Attack
# ============================================

def generate_fgsm_attack(
    model,
    X,
    y,
    epsilon=0.01
):
    """
    Generate epsilon-bounded untargeted FGSM attack.
    """

    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32,
        device=DEVICE,
        requires_grad=True
    )

    y_tensor = torch.tensor(
        y,
        dtype=torch.float32,
        device=DEVICE
    ).view(-1, 1)

    logits = model(X_tensor)

    loss = F.binary_cross_entropy_with_logits(
        logits,
        y_tensor
    )

    model.zero_grad()
    loss.backward()

    gradient = X_tensor.grad.sign()

    # FGSM perturbation
    perturbation = epsilon * gradient

    # Apply perturbation
    X_adv = X_tensor + perturbation

    # Explicitly enforce epsilon bound
    delta = torch.clamp(
        X_adv - X_tensor,
        min=-epsilon,
        max=epsilon
    )

    X_adv = X_tensor + delta

    return X_adv.detach().cpu().numpy()

In [85]:
# ============================================
# STEP 15.1 — Generate Correct FGSM
# ============================================

EPSILON = 0.01

X_attack_adv_fgsm = generate_fgsm_attack(
    ids_model,
    X_attack_clean,
    np.ones(len(X_attack_clean)),
    epsilon=EPSILON
)

perturbation = (
    X_attack_adv_fgsm -
    X_attack_clean
)

print("Original attack shape:",
      X_attack_clean.shape)

print("Adversarial attack shape:",
      X_attack_adv_fgsm.shape)

print(
    "Maximum perturbation:",
    np.max(np.abs(perturbation))
)

print(
    "Mean perturbation:",
    np.mean(np.abs(perturbation))
)

Original attack shape: (11125, 79)
Adversarial attack shape: (11125, 79)
Maximum perturbation: 0.010000079146391894
Mean perturbation: 0.004442247466080784


In [86]:
# ============================================
# STEP 15.2 — Evaluate FGSM Against IDS
# ============================================

def get_predictions(model, X):
    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    ).to(DEVICE)

    with torch.no_grad():
        logits = model(X_tensor)
        probabilities = torch.sigmoid(logits)

    return (
        probabilities.cpu().numpy().flatten()
    )


# Clean predictions
clean_probabilities = get_predictions(
    ids_model,
    X_attack_clean
)

clean_predictions = (
    clean_probabilities >= 0.5
).astype(int)


# Adversarial predictions
adv_probabilities = get_predictions(
    ids_model,
    X_attack_adv_fgsm
)

adv_predictions = (
    adv_probabilities >= 0.5
).astype(int)


# Attack Success Rate
originally_correct = (
    clean_predictions == 1
)

successfully_fooled = (
    adv_predictions == 0
)

attack_success_rate = (
    successfully_fooled[originally_correct].sum()
    / originally_correct.sum()
)


print("FGSM IDS Evaluation")
print("=" * 50)

print(
    f"Clean attack samples correctly classified: "
    f"{originally_correct.sum()}"
)

print(
    f"Adversarial samples classified as benign: "
    f"{successfully_fooled.sum()}"
)

print(
    f"Attack Success Rate: "
    f"{attack_success_rate * 100:.2f}%"
)

FGSM IDS Evaluation
Clean attack samples correctly classified: 10728
Adversarial samples classified as benign: 3723
Attack Success Rate: 31.00%


In [91]:
# ============================================
# Step 15.3 — FGSM → READ Metrics
# ============================================

# Convert FGSM samples to tensor
fgsm_tensor = torch.tensor(
    X_attack_adv_fgsm,
    dtype=torch.float32
).to(DEVICE)

# Extract hidden representation
ids_feature_extractor.eval()

with torch.no_grad():
    fgsm_representation = ids_feature_extractor(fgsm_tensor).cpu().numpy()

print("FGSM adversarial input shape:", X_attack_adv_fgsm.shape)
print("FGSM representation shape:", fgsm_representation.shape)


# --------------------------------------------
# Reconstruct FGSM samples
# --------------------------------------------

fgsm_rep_tensor = torch.tensor(
    fgsm_representation,
    dtype=torch.float32
).to(DEVICE)

reconstruction_model.eval()

with torch.no_grad():
    fgsm_reconstructed = reconstruction_model(
        fgsm_rep_tensor
    ).cpu().numpy()


# --------------------------------------------
# Calculate MedAE
# --------------------------------------------

fgsm_medae = calculate_medae(
    X_attack_adv_fgsm,
    fgsm_reconstructed
)


# --------------------------------------------
# Calculate uncertainty metrics
# --------------------------------------------

fgsm_aleatoric, fgsm_epistemic, fgsm_entropy = (
    calculate_mc_dropout_uncertainty(
        ids_model,
        X_attack_adv_fgsm,
        n_passes=20
    )
)


# --------------------------------------------
# Combine READ metrics
# --------------------------------------------

fgsm_features = np.column_stack([
    fgsm_medae,
    fgsm_aleatoric,
    fgsm_epistemic,
    fgsm_entropy
])


# --------------------------------------------
# Check for NaN / Inf
# --------------------------------------------

print("\nNaN check:")
print("MedAE:", np.isnan(fgsm_medae).sum())
print("Aleatoric:", np.isnan(fgsm_aleatoric).sum())
print("Epistemic:", np.isnan(fgsm_epistemic).sum())
print("Entropy:", np.isnan(fgsm_entropy).sum())


# --------------------------------------------
# Summary
# --------------------------------------------

print("\nFGSM Adversarial READ Metrics")
print("==========================================")
print("Number:", len(fgsm_features))

print(
    f"MedAE      → Mean: {fgsm_medae.mean():.6f}, "
    f"Median: {np.median(fgsm_medae):.6f}, "
    f"Std: {fgsm_medae.std():.6f}"
)

print(
    f"Aleatoric  → Mean: {fgsm_aleatoric.mean():.6f}, "
    f"Median: {np.median(fgsm_aleatoric):.6f}, "
    f"Std: {fgsm_aleatoric.std():.6f}"
)

print(
    f"Epistemic  → Mean: {fgsm_epistemic.mean():.6f}, "
    f"Median: {np.median(fgsm_epistemic):.6f}, "
    f"Std: {fgsm_epistemic.std():.6f}"
)

print(
    f"Entropy    → Mean: {fgsm_entropy.mean():.6f}, "
    f"Median: {np.median(fgsm_entropy):.6f}, "
    f"Std: {fgsm_entropy.std():.6f}"
)

FGSM adversarial input shape: (11125, 79)
FGSM representation shape: (11125, 32)

NaN check:
MedAE: 0
Aleatoric: 0
Epistemic: 0
Entropy: 0

FGSM Adversarial READ Metrics
Number: 11125
MedAE      → Mean: 0.173095, Median: 0.133480, Std: 0.141988
Aleatoric  → Mean: 0.021302, Median: 0.000000, Std: 0.054516
Epistemic  → Mean: 0.007043, Median: 0.000000, Std: 0.021955
Entropy    → Mean: 0.070249, Median: 0.000003, Std: 0.164377


In [90]:
# ============================================
# Step 15.3A — IDS Feature Extractor
# ============================================

class IDSFeatureExtractor(nn.Module):
    def __init__(self, ids_model):
        super().__init__()
        # Everything except the final output layer
        self.feature_layers = nn.Sequential(
            *list(ids_model.network.children())[:-1]
        )

    def forward(self, x):
        return self.feature_layers(x)


ids_feature_extractor = IDSFeatureExtractor(ids_model).to(DEVICE)
ids_feature_extractor.eval()

print("Feature extractor created successfully.")
print(ids_feature_extractor)

Feature extractor created successfully.
IDSFeatureExtractor(
  (feature_layers): Sequential(
    (0): Linear(in_features=79, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
  )
)


In [92]:
# ============================================
# Step 15.4A — READ Detector Training Data
# ============================================

from sklearn.model_selection import train_test_split
import numpy as np

# ------------------------------------------------
# 1. Generate FGSM on reconstruction-training-side
# ------------------------------------------------

# Use correctly classified benign training samples
X_benign_detector = X_benign_correct.cpu().numpy() \
    if torch.is_tensor(X_benign_correct) else X_benign_correct

print("Benign detector source:", X_benign_detector.shape)

# Generate attack labels (all 1)
y_benign_detector = np.zeros(len(X_benign_detector), dtype=int)

# For the detector-training attack samples, use the same number
# of benign samples to keep the classes balanced.
detector_attack_count = len(X_benign_detector)

# Generate FGSM using the IDS
X_detector_fgsm = generate_fgsm_attack(
    ids_model,
    X_benign_detector,
    np.ones(detector_attack_count, dtype=int),
    epsilon=EPSILON
)

print("Generated FGSM samples:", X_detector_fgsm.shape)

Benign detector source: (42682, 79)
Generated FGSM samples: (42682, 79)


In [94]:
# ============================================
# Step 15.4B — READ Features for Detector Train
# ============================================

def get_read_features(X_data):
    """
    Calculate the four READ features:
    MedAE, Aleatoric, Epistemic, Entropy
    """

    X_tensor = torch.tensor(
        X_data,
        dtype=torch.float32
    ).to(DEVICE)

    # Hidden representation
    ids_feature_extractor.eval()

    with torch.no_grad():
        representations = ids_feature_extractor(
            X_tensor
        ).cpu().numpy()

    # Reconstruction
    reconstruction_model.eval()

    rep_tensor = torch.tensor(
        representations,
        dtype=torch.float32
    ).to(DEVICE)

    with torch.no_grad():
        reconstructed = reconstruction_model(
            rep_tensor
        ).cpu().numpy()

    # MedAE
    medae = calculate_medae(
        X_data,
        reconstructed
    )

    # Uncertainty metrics
    aleatoric, epistemic, entropy = (
        calculate_mc_dropout_uncertainty(
            ids_model,
            X_data,
            n_passes=20
        )
    )

    return np.column_stack([
        medae,
        aleatoric,
        epistemic,
        entropy
    ])


# READ features for clean benign detector samples
benign_detector_features = get_read_features(
    X_benign_detector
)

# READ features for FGSM detector samples
fgsm_detector_features = get_read_features(
    X_detector_fgsm
)

print("Benign READ features:", benign_detector_features.shape)
print("FGSM READ features:", fgsm_detector_features.shape)

Benign READ features: (42682, 4)
FGSM READ features: (42682, 4)


In [95]:
# ============================================
# Step 15.4C — READ Detector Split
# ============================================

X_read = np.vstack([
    benign_detector_features,
    fgsm_detector_features
])

y_read = np.concatenate([
    np.zeros(len(benign_detector_features), dtype=int),
    np.ones(len(fgsm_detector_features), dtype=int)
])

# 80% detector training, 20% detector validation
X_read_train, X_read_val, y_read_train, y_read_val = train_test_split(
    X_read,
    y_read,
    test_size=0.20,
    stratify=y_read,
    random_state=SEED
)

print("READ detector training:", X_read_train.shape)
print("READ detector validation:", X_read_val.shape)

print("\nTraining distribution:")
print(np.bincount(y_read_train))

print("\nValidation distribution:")
print(np.bincount(y_read_val))

READ detector training: (68291, 4)
READ detector validation: (17073, 4)

Training distribution:
[34145 34146]

Validation distribution:
[8537 8536]


In [96]:
# ============================================
# Step 15.4D — Train READ Detector
# ============================================

from sklearn.ensemble import RandomForestClassifier

read_detector = RandomForestClassifier(
    n_estimators=200,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced"
)

read_detector.fit(
    X_read_train,
    y_read_train
)

print("READ detector trained successfully.")

READ detector trained successfully.


In [97]:
# ============================================
# Step 15.4E — READ Detector Evaluation
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

read_predictions = read_detector.predict(X_read_val)
read_probabilities = read_detector.predict_proba(X_read_val)[:, 1]

print("READ Detector Evaluation")
print("==========================================")

print(
    "Accuracy :",
    f"{accuracy_score(y_read_val, read_predictions):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_read_val, read_predictions):.4f}"
)

print(
    "Recall   :",
    f"{recall_score(y_read_val, read_predictions):.4f}"
)

print(
    "F1       :",
    f"{f1_score(y_read_val, read_predictions):.4f}"
)

print(
    "ROC-AUC  :",
    f"{roc_auc_score(y_read_val, read_probabilities):.4f}"
)

print(
    "PR-AUC   :",
    f"{average_precision_score(y_read_val, read_probabilities):.4f}"
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_read_val, read_predictions))

READ Detector Evaluation
Accuracy : 0.9527
Precision: 0.9510
Recall   : 0.9547
F1       : 0.9528
ROC-AUC  : 0.9905
PR-AUC   : 0.9900

Confusion Matrix:
[[8117  420]
 [ 387 8149]]


In [98]:
# ============================================
# Step 15.5 — READ Detection on Unseen FGSM
# ============================================

# Clean benign samples from the separate validation set
benign_eval_features = benign_features

# FGSM samples generated from separate IDS validation attacks
fgsm_eval_features = fgsm_features

# Combine evaluation data
X_read_eval = np.vstack([
    benign_eval_features,
    fgsm_eval_features
])

y_read_eval = np.concatenate([
    np.zeros(len(benign_eval_features), dtype=int),
    np.ones(len(fgsm_eval_features), dtype=int)
])

# Predict using the FROZEN READ detector
eval_predictions = read_detector.predict(X_read_eval)
eval_probabilities = read_detector.predict_proba(X_read_eval)[:, 1]

print("Unseen FGSM READ Evaluation")
print("==========================================")

print(
    "Accuracy :",
    f"{accuracy_score(y_read_eval, eval_predictions):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_read_eval, eval_predictions):.4f}"
)

print(
    "Recall   :",
    f"{recall_score(y_read_eval, eval_predictions):.4f}"
)

print(
    "F1       :",
    f"{f1_score(y_read_eval, eval_predictions):.4f}"
)

print(
    "ROC-AUC  :",
    f"{roc_auc_score(y_read_eval, eval_probabilities):.4f}"
)

print(
    "PR-AUC   :",
    f"{average_precision_score(y_read_eval, eval_probabilities):.4f}"
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_read_eval, eval_predictions))

Unseen FGSM READ Evaluation
Accuracy : 0.6441
Precision: 0.9275
Recall   : 0.4024
F1       : 0.5613
ROC-AUC  : 0.8453
PR-AUC   : 0.8675

Confusion Matrix:
[[8187  350]
 [6648 4477]]


In [99]:
# ============================================
# Step 16.1 — BIM Attack
# ============================================

import torch.nn.functional as F

def generate_bim_attack(
    model,
    X,
    y,
    epsilon=0.01,
    alpha=0.002,
    num_iterations=5
):
    model.eval()

    X_original = torch.tensor(
        X,
        dtype=torch.float32,
        device=DEVICE
    )

    y_tensor = torch.tensor(
        y,
        dtype=torch.float32,
        device=DEVICE
    ).view(-1, 1)

    X_adv = X_original.clone().detach()

    for _ in range(num_iterations):

        X_adv.requires_grad_(True)

        logits = model(X_adv)

        loss = F.binary_cross_entropy_with_logits(
            logits,
            y_tensor
        )

        model.zero_grad()

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()

        gradient = X_adv.grad.sign()

        # Iterative FGSM step
        X_adv = X_adv.detach() + alpha * gradient

        # Keep total perturbation within epsilon
        delta = torch.clamp(
            X_adv - X_original,
            min=-epsilon,
            max=epsilon
        )

        X_adv = X_original + delta

    return X_adv.detach().cpu().numpy()

In [100]:
# ============================================
# Step 16.2 — Generate BIM
# ============================================

BIM_EPSILON = 0.01
BIM_ALPHA = 0.002
BIM_ITERATIONS = 5

X_attack_adv_bim = generate_bim_attack(
    ids_model,
    X_attack_clean,
    np.ones(len(X_attack_clean), dtype=int),
    epsilon=BIM_EPSILON,
    alpha=BIM_ALPHA,
    num_iterations=BIM_ITERATIONS
)

perturbation_bim = np.abs(
    X_attack_adv_bim - X_attack_clean
)

print("BIM adversarial shape:", X_attack_adv_bim.shape)
print(
    "Maximum perturbation:",
    perturbation_bim.max()
)
print(
    "Mean perturbation:",
    perturbation_bim.mean()
)

BIM adversarial shape: (11125, 79)
Maximum perturbation: 0.010000034962529991
Mean perturbation: 0.0033756545252025226


In [101]:
# ============================================
# Step 16.3 — BIM IDS Evaluation
# ============================================

clean_probabilities = get_predictions(
    ids_model,
    X_attack_clean
)

clean_predictions = (
    clean_probabilities >= 0.5
).astype(int)

bim_probabilities = get_predictions(
    ids_model,
    X_attack_adv_bim
)

bim_predictions = (
    bim_probabilities >= 0.5
).astype(int)

originally_correct = (
    clean_predictions == 1
)

successfully_fooled = (
    bim_predictions == 0
)

bim_attack_success_rate = (
    successfully_fooled[originally_correct].sum()
    / originally_correct.sum()
)

print("BIM IDS Evaluation")
print("==================================================")
print(
    "Clean attack samples correctly classified:",
    originally_correct.sum()
)
print(
    "Adversarial samples classified as benign:",
    successfully_fooled.sum()
)
print(
    f"Attack Success Rate: "
    f"{bim_attack_success_rate * 100:.2f}%"
)

BIM IDS Evaluation
Clean attack samples correctly classified: 10728
Adversarial samples classified as benign: 3967
Attack Success Rate: 33.28%


In [102]:
# ============================================
# Step 16.4 — BIM READ Metrics
# ============================================

bim_features = get_read_features(
    X_attack_adv_bim
)

bim_medae = bim_features[:, 0]
bim_aleatoric = bim_features[:, 1]
bim_epistemic = bim_features[:, 2]
bim_entropy = bim_features[:, 3]

print("\nBIM Adversarial READ Metrics")
print("==========================================")
print("Number:", len(bim_features))

print(
    f"MedAE      → Mean: {bim_medae.mean():.6f}, "
    f"Median: {np.median(bim_medae):.6f}, "
    f"Std: {bim_medae.std():.6f}"
)

print(
    f"Aleatoric  → Mean: {bim_aleatoric.mean():.6f}, "
    f"Median: {np.median(bim_aleatoric):.6f}, "
    f"Std: {bim_aleatoric.std():.6f}"
)

print(
    f"Epistemic  → Mean: {bim_epistemic.mean():.6f}, "
    f"Median: {np.median(bim_epistemic):.6f}, "
    f"Std: {bim_epistemic.std():.6f}"
)

print(
    f"Entropy    → Mean: {bim_entropy.mean():.6f}, "
    f"Median: {np.median(bim_entropy):.6f}, "
    f"Std: {bim_entropy.std():.6f}"
)


BIM Adversarial READ Metrics
Number: 11125
MedAE      → Mean: 0.174402, Median: 0.133480, Std: 0.140625
Aleatoric  → Mean: 0.011569, Median: 0.000000, Std: 0.040662
Epistemic  → Mean: 0.003075, Median: 0.000000, Std: 0.014688
Entropy    → Mean: 0.039513, Median: 0.000000, Std: 0.122871


In [103]:
# ============================================
# Step 16.5 — BIM READ Detection
# ============================================

X_bim_read_eval = np.vstack([
    benign_features,
    bim_features
])

y_bim_read_eval = np.concatenate([
    np.zeros(len(benign_features), dtype=int),
    np.ones(len(bim_features), dtype=int)
])

bim_read_predictions = read_detector.predict(
    X_bim_read_eval
)

bim_read_probabilities = (
    read_detector.predict_proba(
        X_bim_read_eval
    )[:, 1]
)

print("BIM READ Evaluation")
print("==========================================")

print(
    "Accuracy :",
    f"{accuracy_score(y_bim_read_eval, bim_read_predictions):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_bim_read_eval, bim_read_predictions):.4f}"
)

print(
    "Recall   :",
    f"{recall_score(y_bim_read_eval, bim_read_predictions):.4f}"
)

print(
    "F1       :",
    f"{f1_score(y_bim_read_eval, bim_read_predictions):.4f}"
)

print(
    "ROC-AUC  :",
    f"{roc_auc_score(y_bim_read_eval, bim_read_probabilities):.4f}"
)

print(
    "PR-AUC   :",
    f"{average_precision_score(y_bim_read_eval, bim_read_probabilities):.4f}"
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_bim_read_eval,
        bim_read_predictions
    )
)

BIM READ Evaluation
Accuracy : 0.6908
Precision: 0.9391
Recall   : 0.4849
F1       : 0.6396
ROC-AUC  : 0.8740
PR-AUC   : 0.8952

Confusion Matrix:
[[8187  350]
 [5730 5395]]


In [105]:
# ============================================
# Step 17.1 — PGD Attack
# ============================================

def generate_pgd_attack(
    model,
    X,
    y,
    epsilon=0.01,
    alpha=0.002,
    num_iterations=5
):
    model.eval()

    X_original = torch.tensor(
        X,
        dtype=torch.float32,
        device=DEVICE
    )

    y_tensor = torch.tensor(
        y,
        dtype=torch.float32,
        device=DEVICE
    ).view(-1, 1)

    # Random initialization inside epsilon-ball
    random_delta = torch.empty_like(
        X_original
    ).uniform_(
        -epsilon,
        epsilon
    )

    X_adv = (
        X_original + random_delta
    ).detach()

    for _ in range(num_iterations):

        X_adv.requires_grad_(True)

        logits = model(X_adv)

        loss = F.binary_cross_entropy_with_logits(
            logits,
            y_tensor
        )

        model.zero_grad()

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()

        gradient = X_adv.grad.sign()

        # Gradient ascent
        X_adv = (
            X_adv.detach()
            + alpha * gradient
        )

        # Project back into epsilon-ball
        delta = torch.clamp(
            X_adv - X_original,
            min=-epsilon,
            max=epsilon
        )

        X_adv = (
            X_original + delta
        ).detach()

    return X_adv.cpu().numpy()

In [106]:
# ============================================
# Step 17.2 — Generate PGD
# ============================================

PGD_EPSILON = 0.01
PGD_ALPHA = 0.002
PGD_ITERATIONS = 5

X_attack_adv_pgd = generate_pgd_attack(
    ids_model,
    X_attack_clean,
    np.ones(len(X_attack_clean), dtype=int),
    epsilon=PGD_EPSILON,
    alpha=PGD_ALPHA,
    num_iterations=PGD_ITERATIONS
)

perturbation_pgd = np.abs(
    X_attack_adv_pgd - X_attack_clean
)

print("PGD adversarial shape:", X_attack_adv_pgd.shape)
print(
    "Maximum perturbation:",
    perturbation_pgd.max()
)
print(
    "Mean perturbation:",
    perturbation_pgd.mean()
)

PGD adversarial shape: (11125, 79)
Maximum perturbation: 0.010000077353922499
Mean perturbation: 0.005817134737350296


In [107]:
# ============================================
# Step 17.3 — PGD IDS Evaluation
# ============================================

pgd_probabilities = get_predictions(
    ids_model,
    X_attack_adv_pgd
)

pgd_predictions = (
    pgd_probabilities >= 0.5
).astype(int)

successfully_fooled_pgd = (
    pgd_predictions == 0
)

pgd_attack_success_rate = (
    successfully_fooled_pgd[originally_correct].sum()
    / originally_correct.sum()
)

print("PGD IDS Evaluation")
print("==================================================")
print(
    "Clean attack samples correctly classified:",
    originally_correct.sum()
)
print(
    "Adversarial samples classified as benign:",
    successfully_fooled_pgd.sum()
)
print(
    f"Attack Success Rate: "
    f"{pgd_attack_success_rate * 100:.2f}%"
)

PGD IDS Evaluation
Clean attack samples correctly classified: 10728
Adversarial samples classified as benign: 3439
Attack Success Rate: 28.78%


In [108]:
# ============================================
# Step 17.4 — PGD READ Metrics
# ============================================

pgd_features = get_read_features(
    X_attack_adv_pgd
)

pgd_medae = pgd_features[:, 0]
pgd_aleatoric = pgd_features[:, 1]
pgd_epistemic = pgd_features[:, 2]
pgd_entropy = pgd_features[:, 3]

print("\nPGD Adversarial READ Metrics")
print("==========================================")
print("Number:", len(pgd_features))

print(
    f"MedAE      → Mean: {pgd_medae.mean():.6f}, "
    f"Median: {np.median(pgd_medae):.6f}, "
    f"Std: {pgd_medae.std():.6f}"
)

print(
    f"Aleatoric  → Mean: {pgd_aleatoric.mean():.6f}, "
    f"Median: {np.median(pgd_aleatoric):.6f}, "
    f"Std: {pgd_aleatoric.std():.6f}"
)

print(
    f"Epistemic  → Mean: {pgd_epistemic.mean():.6f}, "
    f"Median: {np.median(pgd_epistemic):.6f}, "
    f"Std: {pgd_epistemic.std():.6f}"
)

print(
    f"Entropy    → Mean: {pgd_entropy.mean():.6f}, "
    f"Median: {np.median(pgd_entropy):.6f}, "
    f"Std: {pgd_entropy.std():.6f}"
)


PGD Adversarial READ Metrics
Number: 11125
MedAE      → Mean: 0.172865, Median: 0.133308, Std: 0.141903
Aleatoric  → Mean: 0.019313, Median: 0.000000, Std: 0.052059
Epistemic  → Mean: 0.004729, Median: 0.000000, Std: 0.017502
Entropy    → Mean: 0.064377, Median: 0.000003, Std: 0.156484


In [109]:
# ============================================
# Step 17.5 — PGD READ Detection
# ============================================

X_pgd_read_eval = np.vstack([
    benign_features,
    pgd_features
])

y_pgd_read_eval = np.concatenate([
    np.zeros(len(benign_features), dtype=int),
    np.ones(len(pgd_features), dtype=int)
])

pgd_read_predictions = read_detector.predict(
    X_pgd_read_eval
)

pgd_read_probabilities = (
    read_detector.predict_proba(
        X_pgd_read_eval
    )[:, 1]
)

print("PGD READ Evaluation")
print("==========================================")

print(
    "Accuracy :",
    f"{accuracy_score(y_pgd_read_eval, pgd_read_predictions):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_pgd_read_eval, pgd_read_predictions):.4f}"
)

print(
    "Recall   :",
    f"{recall_score(y_pgd_read_eval, pgd_read_predictions):.4f}"
)

print(
    "F1       :",
    f"{f1_score(y_pgd_read_eval, pgd_read_predictions):.4f}"
)

print(
    "ROC-AUC  :",
    f"{roc_auc_score(y_pgd_read_eval, pgd_read_probabilities):.4f}"
)

print(
    "PR-AUC   :",
    f"{average_precision_score(y_pgd_read_eval, pgd_read_probabilities):.4f}"
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_pgd_read_eval,
        pgd_read_predictions
    )
)

PGD READ Evaluation
Accuracy : 0.6495
Precision: 0.9290
Recall   : 0.4120
F1       : 0.5708
ROC-AUC  : 0.8550
PR-AUC   : 0.8739

Confusion Matrix:
[[8187  350]
 [6542 4583]]


In [110]:
# ============================================
# Step 18 — Comparison Table
# ============================================

import pandas as pd

# Initialize a dictionary to store metrics
comparison_metrics = {}

# --- FGSM Metrics ---
comparison_metrics['FGSM'] = {
    'Attack Success Rate (IDS)': f"{attack_success_rate * 100:.2f}%",
    'READ Detector Accuracy': f"{accuracy_score(y_read_eval, eval_predictions):.4f}",
    'READ Detector Precision': f"{precision_score(y_read_eval, eval_predictions):.4f}",
    'READ Detector Recall': f"{recall_score(y_read_eval, eval_predictions):.4f}",
    'READ Detector F1': f"{f1_score(y_read_eval, eval_predictions):.4f}",
    'READ Detector ROC-AUC': f"{roc_auc_score(y_read_eval, eval_probabilities):.4f}",
    'READ Detector PR-AUC': f"{average_precision_score(y_read_eval, eval_probabilities):.4f}",
    'MedAE Mean': f"{fgsm_medae.mean():.6f}",
    'MedAE Median': f"{np.median(fgsm_medae):.6f}",
    'MedAE Std': f"{fgsm_medae.std():.6f}",
    'Aleatoric Mean': f"{fgsm_aleatoric.mean():.6f}",
    'Aleatoric Median': f"{np.median(fgsm_aleatoric):.6f}",
    'Aleatoric Std': f"{fgsm_aleatoric.std():.6f}",
    'Epistemic Mean': f"{fgsm_epistemic.mean():.6f}",
    'Epistemic Median': f"{np.median(fgsm_epistemic):.6f}",
    'Epistemic Std': f"{fgsm_epistemic.std():.6f}",
    'Entropy Mean': f"{fgsm_entropy.mean():.6f}",
    'Entropy Median': f"{np.median(fgsm_entropy):.6f}",
    'Entropy Std': f"{fgsm_entropy.std():.6f}"
}

# --- BIM Metrics ---
comparison_metrics['BIM'] = {
    'Attack Success Rate (IDS)': f"{bim_attack_success_rate * 100:.2f}%",
    'READ Detector Accuracy': f"{accuracy_score(y_bim_read_eval, bim_read_predictions):.4f}",
    'READ Detector Precision': f"{precision_score(y_bim_read_eval, bim_read_predictions):.4f}",
    'READ Detector Recall': f"{recall_score(y_bim_read_eval, bim_read_predictions):.4f}",
    'READ Detector F1': f"{f1_score(y_bim_read_eval, bim_read_predictions):.4f}",
    'READ Detector ROC-AUC': f"{roc_auc_score(y_bim_read_eval, bim_read_probabilities):.4f}",
    'READ Detector PR-AUC': f"{average_precision_score(y_bim_read_eval, bim_read_probabilities):.4f}",
    'MedAE Mean': f"{bim_medae.mean():.6f}",
    'MedAE Median': f"{np.median(bim_medae):.6f}",
    'MedAE Std': f"{bim_medae.std():.6f}",
    'Aleatoric Mean': f"{bim_aleatoric.mean():.6f}",
    'Aleatoric Median': f"{np.median(bim_aleatoric):.6f}",
    'Aleatoric Std': f"{bim_aleatoric.std():.6f}",
    'Epistemic Mean': f"{bim_epistemic.mean():.6f}",
    'Epistemic Median': f"{np.median(bim_epistemic):.6f}",
    'Epistemic Std': f"{bim_epistemic.std():.6f}",
    'Entropy Mean': f"{bim_entropy.mean():.6f}",
    'Entropy Median': f"{np.median(bim_entropy):.6f}",
    'Entropy Std': f"{bim_entropy.std():.6f}"
}

# --- PGD Metrics ---
comparison_metrics['PGD'] = {
    'Attack Success Rate (IDS)': f"{pgd_attack_success_rate * 100:.2f}%",
    'READ Detector Accuracy': f"{accuracy_score(y_pgd_read_eval, pgd_read_predictions):.4f}",
    'READ Detector Precision': f"{precision_score(y_pgd_read_eval, pgd_read_predictions):.4f}",
    'READ Detector Recall': f"{recall_score(y_pgd_read_eval, pgd_read_predictions):.4f}",
    'READ Detector F1': f"{f1_score(y_pgd_read_eval, pgd_read_predictions):.4f}",
    'READ Detector ROC-AUC': f"{roc_auc_score(y_pgd_read_eval, pgd_read_probabilities):.4f}",
    'READ Detector PR-AUC': f"{average_precision_score(y_pgd_read_eval, pgd_read_probabilities):.4f}",
    'MedAE Mean': f"{pgd_medae.mean():.6f}",
    'MedAE Median': f"{np.median(pgd_medae):.6f}",
    'MedAE Std': f"{pgd_medae.std():.6f}",
    'Aleatoric Mean': f"{pgd_aleatoric.mean():.6f}",
    'Aleatoric Median': f"{np.median(pgd_aleatoric):.6f}",
    'Aleatoric Std': f"{pgd_aleatoric.std():.6f}",
    'Epistemic Mean': f"{pgd_epistemic.mean():.6f}",
    'Epistemic Median': f"{np.median(pgd_epistemic):.6f}",
    'Epistemic Std': f"{pgd_epistemic.std():.6f}",
    'Entropy Mean': f"{pgd_entropy.mean():.6f}",
    'Entropy Median': f"{np.median(pgd_entropy):.6f}",
    'Entropy Std': f"{pgd_entropy.std():.6f}"
}

# Create DataFrame
comparison_df = pd.DataFrame.from_dict(
    comparison_metrics,
    orient='index'
)

display(comparison_df)

,Attack Success Rate (IDS),READ Detector Accuracy,READ Detector Precision,READ Detector Recall,READ Detector F1,READ Detector ROC-AUC,READ Detector PR-AUC,MedAE Mean,MedAE Median,MedAE Std,Aleatoric Mean,Aleatoric Median,Aleatoric Std,Epistemic Mean,Epistemic Median,Epistemic Std,Entropy Mean,Entropy Median,Entropy Std
FGSM,31.00%,0.6441,0.9275,0.4024,0.5613,0.8453,0.8675,0.173095,0.133480,0.141988,0.021302,0.000000,0.054516,0.007043,0.000000,0.021955,0.070249,0.000003,0.164377
BIM,33.28%,0.6908,0.9391,0.4849,0.6396,0.8740,0.8952,0.174402,0.133480,0.140625,0.011569,0.000000,0.040662,0.003075,0.000000,0.014688,0.039513,0.000000,0.122871
PGD,28.78%,0.6495,0.9290,0.4120,0.5708,0.8550,0.8739,0.172865,0.133308,0.141903,0.019313,0.000000,0.052059,0.004729,0.000000,0.017502,0.064377,0.000003,0.156484
